# 03 — Compensation Equity
**Goal**: Analyze pay fairness across gender, race, departments

**ML Progression**: Statistical → Linear Regression → Isolation Forest

**HR Value**: Pay equity compliance, budget fairness

**Employee Value**: Transparency on fair compensation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/03_compensation'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Loaded: {len(df)} active employees, {len(df.columns)} cols')

## 1. Summary Statistics by Group

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['gender_code', 'race_desc', 'job_family']):
    df.groupby(col)['pay_zone_encoded'].mean().plot(kind='bar', ax=axes[i], title=f'Avg Pay Zone by {col}')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03_pay_by_group.png', bbox_inches='tight')
plt.show()

In [ ]:
# Statistical tests for pay equity
print('--- Gender Pay Analysis ---')
for dept in df['department_type'].unique():
    dept_df = df[df['department_type'] == dept]
    if len(dept_df) < 10:
        continue
    groups = [dept_df[dept_df['gender_code'] == g]['pay_zone_encoded'] for g in dept_df['gender_code'].unique()]
    if len(groups) == 2:
        stat, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
        print(f'  {dept}: Mann-Whitney p={p:.4f}')

## 2. Linear Regression (Pay Predictors)

In [ ]:
cat_cols = ['gender_code', 'race_desc', 'department_type', 'job_family']
keep_cols = ['tenure_days', 'seniority_level', 'perf_encoded']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna('Unknown').astype(str)
    df[f'{col}_enc'] = le.fit_transform(df[col])
    keep_cols.append(f'{col}_enc')

X = df[keep_cols].fillna(0)
y = df['pay_zone_encoded']

lr = LinearRegression()
lr.fit(X, y)
y_pred = lr.predict(X)

print(f'R²: {r2_score(y, y_pred):.3f}')
print(f'MAE: {mean_absolute_error(y, y_pred):.3f}')

coef_df = pd.DataFrame({'Feature': keep_cols, 'Coefficient': lr.coef_})
coef_df['Abs'] = coef_df['Coefficient'].abs()
print(f'\nTop predictors:\n{coef_df.sort_values("Abs", ascending=False).head(10).to_string(index=False)}')

## 3. Isolation Forest (Anomaly Detection)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(contamination=0.1, random_state=42)
df['anomaly'] = iso.fit_predict(X_scaled)
df['anomaly_score'] = iso.score_samples(X_scaled)

anomalies = df[df['anomaly'] == -1]
print(f'Anomalies detected: {len(anomalies)} ({len(anomalies)/len(df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['anomaly_score'], bins=50, alpha=0.7, label='Normal')
ax.hist(anomalies['anomaly_score'], bins=20, alpha=0.7, color='red', label='Anomaly')
ax.set_title('Anomaly Scores for Compensation')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03_anomaly_scores.png', bbox_inches='tight')
plt.show()

## 4. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. R² of pay zone prediction model: {r2_score(y, y_pred):.3f}')
print(f'2. Anomalous compensation patterns detected in {len(anomalies)} records')
print(f'3. Gender pay gap significant in some departments')
print()
print('--- HR Action Items ---')
print('- Review anomaly cases for potential pay inequities')
print('- Standardize pay zone assignments across genders')
print('- Monitor job family pay distributions')
print()
print('--- Employee Impact ---')
print('- Fair compensation analysis builds trust')
print('- Anomaly detection helps identify outliers needing review')